# E070 -- JWST Galaxy Evolution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e070_jwst_galaxies.ipynb)

**Objective:** Download a JWST photometric catalog (FITS format), extract photometric redshifts and galaxy sizes, and analyze galaxy evolution — how galaxy properties change with cosmic time.

JWST's unprecedented infrared sensitivity allows us to detect galaxies at redshifts z > 10 (less than 500 million years after the Big Bang), probing the epoch of first galaxy formation.

## Data Source

- **Instrument:** James Webb Space Telescope (JWST) NIRCam
- **Catalog:** JWST photometric redshift catalog (FITS format)
- **Download:** Google Drive hosted copy
- **Fields used:** z_phot (photometric redshift), flux_radius (half-light radius)
- **Requires:** `astropy` for FITS file handling

In [ ]:
# Install astropy if needed (for FITS file handling)
!pip install -q astropy gdown

In [ ]:
import subprocess
import os
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from astropy.io import fits

# Download JWST catalog from Google Drive
file_id = "19Jd3Xg5BTrHX5icb81txSn_P0LqxmLsz"
output_file = "jwst_catalog.fits"

if not os.path.exists(output_file):
    import gdown
    url = f"https://drive.google.com/uc?id={file_id}"
    print("Downloading JWST catalog from Google Drive...")
    gdown.download(url, output_file, quiet=False)
else:
    print(f"Using cached {output_file}")

# Load FITS catalog
hdulist = fits.open(output_file)
print(f"HDU list: {[h.name for h in hdulist]}")
catalog = hdulist[1].data
cols = hdulist[1].columns.names
print(f"Columns ({len(cols)}): {cols[:20]}")
print(f"Total objects: {len(catalog)}")

In [ ]:
# Extract photometric redshifts and sizes
# Column names vary by catalog — search for relevant ones
z_col = None
r_col = None
mag_col = None

for c in cols:
    cl = c.lower()
    if z_col is None and ('z_phot' in cl or 'photo_z' in cl or cl == 'z' or 'redshift' in cl):
        z_col = c
    if r_col is None and ('flux_radius' in cl or 'half_light' in cl or 're' == cl or 'kron_radius' in cl):
        r_col = c
    if mag_col is None and ('mag' in cl and 'auto' in cl):
        mag_col = c

print(f"Redshift column: {z_col}")
print(f"Radius column: {r_col}")
print(f"Magnitude column: {mag_col}")

z_phot = catalog[z_col] if z_col else None
flux_r = catalog[r_col] if r_col else None

# Filter valid data
if z_phot is not None:
    valid = np.isfinite(z_phot) & (z_phot > 0) & (z_phot < 20)
    if flux_r is not None:
        valid &= np.isfinite(flux_r) & (flux_r > 0)
    z_phot_clean = z_phot[valid]
    flux_r_clean = flux_r[valid] if flux_r is not None else None
    print(f"\nValid galaxies: {valid.sum()}")
    print(f"Redshift range: [{z_phot_clean.min():.2f}, {z_phot_clean.max():.2f}]")
    if flux_r_clean is not None:
        print(f"Flux radius range: [{flux_r_clean.min():.2f}, {flux_r_clean.max():.2f}] pixels")

## Redshift Distribution

The photometric redshift distribution reveals the depth of the JWST survey. Higher redshift means we are looking further back in time — z=1 corresponds to ~8 billion years ago, z=6 to ~12.8 billion years ago.

In [ ]:
# Compute lookback time from redshift (simplified flat LCDM)
from scipy.integrate import quad

H0 = 70.0  # km/s/Mpc
Om = 0.3
OL = 0.7
t_H = 1.0 / H0 * 3.086e19 / 3.156e16  # Hubble time in Gyr

def integrand(z):
    return 1.0 / ((1 + z) * np.sqrt(Om * (1 + z)**3 + OL))

def lookback_time(z):
    val, _ = quad(integrand, 0, z)
    return t_H * val

# Compute for binned redshifts
z_ticks = [0.5, 1, 2, 3, 5, 8, 10]
t_ticks = [lookback_time(z) for z in z_ticks]
print("Redshift → Lookback Time:")
for z, t in zip(z_ticks, t_ticks):
    print(f"  z = {z:>4} → {t:.1f} Gyr ago")

In [ ]:
# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E070: JWST Galaxy Evolution",
             fontsize=15, fontweight='bold')

# (a) Redshift distribution
ax = axes[0, 0]
ax.hist(z_phot_clean, bins=80, range=(0, 12), color='steelblue',
        edgecolor='black', linewidth=0.3, alpha=0.8)
ax.set_xlabel('Photometric Redshift z', fontsize=11)
ax.set_ylabel('Number of Galaxies', fontsize=11)
ax.set_title(f'(a) Redshift Distribution (N={len(z_phot_clean)})', fontsize=12)
# Add lookback time axis on top
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(z_ticks[:5])
ax2.set_xticklabels([f'{t:.0f}' for t in t_ticks[:5]], fontsize=9)
ax2.set_xlabel('Lookback Time [Gyr]', fontsize=10)

# (b) Size vs redshift
ax = axes[0, 1]
if flux_r_clean is not None:
    ax.scatter(z_phot_clean, flux_r_clean, s=1, alpha=0.1, color='teal')
    # Binned medians
    z_edges = np.arange(0, 10, 0.5)
    z_mids, r_meds, r_q25, r_q75 = [], [], [], []
    for i in range(len(z_edges) - 1):
        m = (z_phot_clean >= z_edges[i]) & (z_phot_clean < z_edges[i + 1])
        if m.sum() > 10:
            z_mids.append((z_edges[i] + z_edges[i + 1]) / 2)
            r_meds.append(np.median(flux_r_clean[m]))
            r_q25.append(np.percentile(flux_r_clean[m], 25))
            r_q75.append(np.percentile(flux_r_clean[m], 75))
    z_mids = np.array(z_mids)
    r_meds = np.array(r_meds)
    ax.plot(z_mids, r_meds, 'r-o', markersize=4, linewidth=2, label='Median')
    ax.fill_between(z_mids, r_q25, r_q75, alpha=0.2, color='red', label='IQR')
    ax.set_xlabel('Photometric Redshift z', fontsize=11)
    ax.set_ylabel('Flux Radius [pixels]', fontsize=11)
    ax.set_title('(b) Galaxy Size vs Redshift', fontsize=12)
    ax.legend(fontsize=10)
    ax.set_ylim(0, np.percentile(flux_r_clean, 95) * 1.5)
else:
    ax.text(0.5, 0.5, 'No size data available', transform=ax.transAxes, ha='center')

# (c) Cumulative redshift distribution
ax = axes[1, 0]
z_sorted = np.sort(z_phot_clean)
cdf = np.arange(1, len(z_sorted) + 1) / len(z_sorted)
ax.plot(z_sorted, cdf, 'steelblue', linewidth=2)
for z_mark in [1, 2, 4, 6]:
    frac = np.mean(z_phot_clean > z_mark)
    ax.axvline(z_mark, color='gray', linestyle=':', alpha=0.5)
    ax.annotate(f'{frac*100:.1f}% at z>{z_mark}', xy=(z_mark, 1 - frac),
                fontsize=8, ha='left')
ax.set_xlabel('Photometric Redshift z', fontsize=11)
ax.set_ylabel('Cumulative Fraction', fontsize=11)
ax.set_title('(c) Cumulative Redshift Distribution', fontsize=12)

# (d) Number density per redshift bin
ax = axes[1, 1]
z_centers_hist = []
n_per_bin = []
dz = 0.5
for z_lo in np.arange(0, 12, dz):
    n = np.sum((z_phot_clean >= z_lo) & (z_phot_clean < z_lo + dz))
    z_centers_hist.append(z_lo + dz / 2)
    n_per_bin.append(n)
ax.bar(z_centers_hist, n_per_bin, width=dz * 0.9, color='coral', edgecolor='black', linewidth=0.3)
ax.set_xlabel('Redshift z', fontsize=11)
ax.set_ylabel('Galaxy Count per Δz=0.5 bin', fontsize=11)
ax.set_title('(d) Galaxy Number Density vs Redshift', fontsize=12)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('e070_jwst_galaxies.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e070_jwst_galaxies.png")

## Key Results Summary

| Analysis | Result |
|----------|--------|
| Total galaxies in catalog | From JWST deep field FITS |
| Redshift range | z ~ 0 to z > 8 (first billion years) |
| Size evolution | Galaxies are smaller at higher redshift |
| High-z fraction | Percentage of galaxies beyond z > 4 |

**Physical interpretation:** JWST reveals that galaxies in the early universe were significantly smaller and more compact than their present-day counterparts. The size evolution with redshift traces the hierarchical assembly of galaxies — small building blocks merge over cosmic time to form the large galaxies we see today. The detection of substantial galaxy populations at z > 6 challenges some models of early galaxy formation.